# Document Intelligence Assistant
Shashank Pandey · code: https://github.com/Shashank4355/AI-ML-Engineer-Assignment

RAG + agentic Q&A over 12 arXiv papers, with the model (Qwen2.5-VL-7B-Instruct AWQ) served by vLLM. Answers include file/page citations, and when the papers don't have the answer it says so.

This notebook writes the project files, installs everything, runs all the checks, then runs a short demo and collects the evaluation answers. It needs GPU T4 x2 and Internet turned on, and takes about 20 min.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Project files
One cell per file, the same as in the repo.

In [ ]:
import os
os.makedirs('/kaggle/working/dia/app', exist_ok=True)
os.makedirs('/kaggle/working/dia/scripts', exist_ok=True)
os.makedirs('/kaggle/working/dia/eval', exist_ok=True)
%cd /kaggle/working/dia
open('app/__init__.py', 'w').close()

### `requirements.txt`

In [ ]:
%%writefile requirements.txt
fastapi
uvicorn[standard]
python-multipart
pydantic>=2.9
openai>=1.55
langgraph>=0.2.60
qdrant-client>=1.12
sentence-transformers>=3.3
rank-bm25
pymupdf>=1.25
python-dotenv>=1.0
requests>=2.32
google-genai>=1.0


### `app/config.py` - settings

In [ ]:
%%writefile app/config.py
import os
from dotenv import load_dotenv

load_dotenv()

VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_MODEL = os.getenv("VLLM_MODEL", "Qwen/Qwen2.5-VL-7B-Instruct-AWQ")
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
EMBED_MODEL = os.getenv("EMBED_MODEL", "BAAI/bge-base-en-v1.5")
COLLECTION = "docs"
PDF_DIR = os.getenv("PDF_DIR", "data/pdfs")
BM25_PATH = os.getenv("BM25_PATH", "data/bm25.pkl")

CHUNK_SIZE = 800      # characters (~200 tokens)
CHUNK_OVERLAP = 150
TOP_K = 5
MAX_RETRIES = 2


### `app/llm.py` - vLLM client (text + image)

In [ ]:
%%writefile app/llm.py
"""Thin client for the self-hosted vLLM server (OpenAI-compatible API)."""
import base64
import io

from PIL import Image
from openai import OpenAI
from app.config import VLLM_BASE_URL, VLLM_MODEL

client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed")


def chat(prompt: str, system: str = "You are a helpful assistant.", max_tokens: int = 512) -> str:
    resp = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()


def _shrink(image_bytes: bytes, max_side: int = 768) -> bytes:
    """Downscale large images so their vision tokens fit in the context window."""
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def chat_image(image_bytes: bytes, prompt: str, mime: str = "image/png", max_tokens: int = 512) -> str:
    b64 = base64.b64encode(_shrink(image_bytes)).decode()
    mime = "image/png"
    resp = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
                {"type": "text", "text": prompt},
            ],
        }],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()


### `app/ingest.py` - PDF parsing, chunking, figure descriptions, indexing

In [ ]:
%%writefile app/ingest.py
"""Offline pipeline: parse PDFs -> chunk -> (VLM figure captions) -> embed -> Qdrant + BM25.

Run:  python -m app.ingest            (text + figures)
      python -m app.ingest --no-images (text only, faster)
"""
import argparse
import pickle
import uuid
from pathlib import Path

import pymupdf as fitz
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from app.config import (BM25_PATH, CHUNK_OVERLAP, CHUNK_SIZE, COLLECTION,
                        EMBED_MODEL, PDF_DIR, QDRANT_URL)
from app.llm import chat_image

FIGURE_PROMPT = (
    "Describe this figure from a research paper in detail: its type (chart, diagram, table), "
    "every label, component, axis, value and relationship shown. Be factual and specific."
)


def chunk_text(text: str) -> list[str]:
    text = " ".join(text.split())
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + CHUNK_SIZE])
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return [c for c in chunks if len(c) > 50]


def extract(pdf_path: Path, with_images: bool) -> list[dict]:
    doc = fitz.open(pdf_path)
    records = []
    for page_no, page in enumerate(doc, start=1):
        for chunk in chunk_text(page.get_text()):
            records.append({"text": chunk, "source": pdf_path.name, "page": page_no, "type": "text"})

        if not with_images:
            continue
        for img in page.get_images(full=True):
            pix = fitz.Pixmap(doc, img[0])
            if pix.width < 200 or pix.height < 150:  # skip icons/logos
                continue
            if pix.n - pix.alpha >= 4:
                pix = fitz.Pixmap(fitz.csRGB, pix)
            try:
                desc = chat_image(pix.tobytes("png"), FIGURE_PROMPT)
            except Exception as e:
                print(f"  ! figure caption failed on {pdf_path.name} p{page_no}: {e}")
                continue
            records.append({"text": f"[Figure] {desc}", "source": pdf_path.name,
                            "page": page_no, "type": "figure"})
    return records


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--no-images", action="store_true")
    args = ap.parse_args()

    records = []
    for pdf in sorted(Path(PDF_DIR).glob("*.pdf")):
        print(f"Parsing {pdf.name}")
        records += extract(pdf, with_images=not args.no_images)
    print(f"{len(records)} chunks ({sum(r['type'] == 'figure' for r in records)} figures)")

    model = SentenceTransformer(EMBED_MODEL)
    vectors = model.encode([r["text"] for r in records], normalize_embeddings=True,
                           batch_size=32, show_progress_bar=True)

    qdrant = QdrantClient(url=QDRANT_URL)
    qdrant.recreate_collection(COLLECTION, vectors_config=VectorParams(size=vectors.shape[1],
                                                                       distance=Distance.COSINE))
    points = [PointStruct(id=str(uuid.uuid4()), vector=v.tolist(), payload=r)
              for v, r in zip(vectors, records)]
    for i in range(0, len(points), 256):
        qdrant.upsert(COLLECTION, points[i:i + 256])

    bm25 = BM25Okapi([r["text"].lower().split() for r in records])
    Path(BM25_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(BM25_PATH, "wb") as f:
        pickle.dump({"bm25": bm25, "records": records}, f)
    print("Indexing done.")


if __name__ == "__main__":
    main()


### `app/retriever.py` - dense + BM25 retrieval

In [ ]:
%%writefile app/retriever.py
"""Hybrid retrieval: dense (bge + Qdrant) + sparse (BM25), fused with Reciprocal Rank Fusion."""
import pickle
from functools import lru_cache

from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

from app.config import BM25_PATH, COLLECTION, EMBED_MODEL, QDRANT_URL, TOP_K

BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


@lru_cache
def _resources():
    with open(BM25_PATH, "rb") as f:
        bm25 = pickle.load(f)
    return SentenceTransformer(EMBED_MODEL), QdrantClient(url=QDRANT_URL), bm25


def _key(r: dict) -> str:
    return f"{r['source']}|{r['page']}|{r['text'][:60]}"


def retrieve(query: str, k: int = TOP_K) -> list[dict]:
    model, qdrant, bm = _resources()

    vec = model.encode(BGE_QUERY_PREFIX + query, normalize_embeddings=True).tolist()
    dense = [p.payload for p in qdrant.query_points(COLLECTION, query=vec, limit=k * 3).points]

    scores = bm["bm25"].get_scores(query.lower().split())
    top = sorted(range(len(scores)), key=lambda i: -scores[i])[:k * 3]
    sparse = [bm["records"][i] for i in top]

    fused, docs = {}, {}
    for ranking in (dense, sparse):
        for rank, r in enumerate(ranking):
            key = _key(r)
            fused[key] = fused.get(key, 0) + 1 / (60 + rank)
            docs[key] = r
    return [docs[key] for key in sorted(fused, key=fused.get, reverse=True)[:k]]


### `app/graph.py` - LangGraph agent

In [ ]:
%%writefile app/graph.py
"""Agentic workflow (LangGraph):

router -> [chitchat | out_of_scope | retrieve]
retrieve -> grade -> (relevant ? answer : rewrite -> retrieve, max MAX_RETRIES) / fallback
answer -> verify -> (supported ? END : fallback)
"""
from typing import TypedDict

from langgraph.graph import END, StateGraph

from app.config import MAX_RETRIES
from app.llm import chat
from app.retriever import retrieve

NOT_FOUND = "I couldn't find this in the provided documents."


class State(TypedDict, total=False):
    question: str
    query: str
    route: str
    chunks: list[dict]
    retries: int
    answer: str
    sources: list[dict]
    path: list[str]


def _step(state: State, name: str) -> list[str]:
    return state.get("path", []) + [name]


def router(state: State) -> State:
    label = chat(
        "Classify the user message into exactly one label:\n"
        "document - a question that could be answered from a corpus of AI/ML research papers "
        "(LLMs, transformers, RAG, retrieval, fine-tuning, model serving)\n"
        "chitchat - greetings, thanks, small talk\n"
        "out_of_scope - anything else (sports, news, general trivia, etc.)\n\n"
        f"Message: {state['question']}\nLabel:", max_tokens=5).lower()
    route = next((r for r in ("document", "chitchat", "out_of_scope") if r in label), "document")
    return {"route": route, "query": state["question"], "retries": 0, "path": _step(state, f"router:{route}")}


def chitchat(state: State) -> State:
    ans = chat(state["question"], system="You are a friendly assistant for a research-paper Q&A tool. "
               "Reply briefly and invite a question about the documents.", max_tokens=100)
    return {"answer": ans, "sources": [], "path": _step(state, "chitchat")}


def fallback(state: State) -> State:
    return {"answer": NOT_FOUND, "sources": [], "path": _step(state, "fallback")}


def retrieve_node(state: State) -> State:
    return {"chunks": retrieve(state["query"]), "path": _step(state, "retrieve")}


def grade(state: State) -> State:
    relevant = []
    for c in state["chunks"]:
        verdict = chat(f"Question: {state['question']}\n\nPassage: {c['text']}\n\n"
                       "Does this passage contain information useful to answer the question? "
                       "Answer yes or no.", max_tokens=3).lower()
        if verdict.startswith("yes"):
            relevant.append(c)
    return {"chunks": relevant, "path": _step(state, f"grade:{len(relevant)}_relevant")}


def rewrite(state: State) -> State:
    q = chat(f"Rewrite this question as a better search query for research papers. "
             f"Return only the query.\n\nQuestion: {state['question']}", max_tokens=60)
    return {"query": q, "retries": state["retries"] + 1, "path": _step(state, "rewrite")}


def _context(chunks: list[dict]) -> str:
    return "\n\n".join(f"[{i + 1}] (source: {c['source']}, page {c['page']})\n{c['text']}"
                       for i, c in enumerate(chunks))


def answer(state: State) -> State:
    ans = chat(
        f"Context:\n{_context(state['chunks'])}\n\nQuestion: {state['question']}\n\n"
        "Answer using ONLY the context. Cite every claim as [Source: <file>, page <n>]. "
        f"If the context does not contain the answer, reply exactly: {NOT_FOUND}",
        system="You are a precise assistant that answers strictly from provided documents.")
    sources = [{"source": c["source"], "page": c["page"], "type": c["type"]} for c in state["chunks"]]
    return {"answer": ans, "sources": sources, "path": _step(state, "answer")}


def verify(state: State) -> State:
    if NOT_FOUND in state["answer"]:
        return {"route": "unsupported", "path": _step(state, "verify:not_found")}
    verdict = chat(f"Context:\n{_context(state['chunks'])}\n\nAnswer: {state['answer']}\n\n"
                   "Is every claim in the answer supported by the context? Answer yes or no.",
                   max_tokens=3).lower()
    ok = verdict.startswith("yes")
    return {"route": "supported" if ok else "unsupported", "path": _step(state, f"verify:{'pass' if ok else 'fail'}")}


def after_grade(state: State) -> str:
    if state["chunks"]:
        return "answer"
    return "rewrite" if state["retries"] < MAX_RETRIES else "fallback"


def build_graph():
    g = StateGraph(State)
    for name, fn in [("router", router), ("chitchat", chitchat), ("fallback", fallback),
                     ("retrieve", retrieve_node), ("grade", grade), ("rewrite", rewrite),
                     ("answer", answer), ("verify", verify)]:
        g.add_node(name, fn)
    g.set_entry_point("router")
    g.add_conditional_edges("router", lambda s: s["route"],
                            {"document": "retrieve", "chitchat": "chitchat", "out_of_scope": "fallback"})
    g.add_edge("retrieve", "grade")
    g.add_conditional_edges("grade", after_grade)
    g.add_edge("rewrite", "retrieve")
    g.add_edge("answer", "verify")
    g.add_conditional_edges("verify", lambda s: END if s["route"] == "supported" else "fallback")
    g.add_edge("chitchat", END)
    g.add_edge("fallback", END)
    return g.compile()


agent = build_graph()


### `app/main.py` - FastAPI app

In [ ]:
%%writefile app/main.py
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from pydantic import BaseModel

from app.config import VLLM_MODEL
from app.graph import agent
from app.llm import chat_image, client

app = FastAPI(title="Document Intelligence Assistant")


class AskRequest(BaseModel):
    question: str


class AskResponse(BaseModel):
    answer: str
    sources: list[dict]
    agent_path: list[str]


@app.get("/health")
def health():
    try:
        models = [m.id for m in client.models.list().data]
        return {"status": "ok", "vllm_models": models}
    except Exception as e:
        return {"status": "degraded", "error": str(e), "expected_model": VLLM_MODEL}


@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    if not req.question.strip():
        raise HTTPException(400, "question must not be empty")
    try:
        out = agent.invoke({"question": req.question})
    except Exception as e:
        raise HTTPException(503, f"Backend error: {e}")
    return AskResponse(answer=out["answer"], sources=out.get("sources", []), agent_path=out["path"])


@app.post("/ask-image")
async def ask_image(question: str = Form(...), image: UploadFile = File(...)):
    data = await image.read()
    try:
        ans = chat_image(data, question, mime=image.content_type or "image/png")
    except Exception as e:
        raise HTTPException(503, f"Backend error: {e}")
    return {"answer": ans, "agent_path": ["visual_qa"]}


### `scripts/download_docs.py` - downloads the papers

In [ ]:
%%writefile scripts/download_docs.py
"""Download the knowledge base: 12 public arXiv papers on LLMs / RAG / serving."""
from pathlib import Path
import urllib.request

PAPERS = {
    "attention_is_all_you_need": "1706.03762",
    "bert": "1810.04805",
    "rag_lewis": "2005.11401",
    "dense_passage_retrieval": "2004.04906",
    "lora": "2106.09685",
    "chain_of_thought": "2201.11903",
    "react": "2210.03629",
    "vllm_pagedattention": "2309.06180",
    "self_rag": "2310.11511",
    "corrective_rag": "2401.15884",
    "bge_m3": "2402.03216",
    "qwen2_5_vl": "2502.13923",
}

out = Path("data/pdfs")
out.mkdir(parents=True, exist_ok=True)
for name, arxiv_id in PAPERS.items():
    dest = out / f"{name}.pdf"
    if dest.exists():
        continue
    print(f"Downloading {name}")
    req = urllib.request.Request(f"https://arxiv.org/pdf/{arxiv_id}", headers={"User-Agent": "Mozilla/5.0"})
    dest.write_bytes(urllib.request.urlopen(req).read())
print("Done.")


### `scripts/benchmark.py` - load test

In [ ]:
%%writefile scripts/benchmark.py
"""Concurrent load test against vLLM: reports TTFT, P95 latency and output tokens/sec.

python scripts/benchmark.py --concurrency 32
"""
import argparse
import asyncio
import os
import statistics
import time

from openai import AsyncOpenAI

BASE = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
MODEL = os.getenv("VLLM_MODEL", "Qwen/Qwen2.5-VL-7B-Instruct-AWQ")
PROMPT = "Explain retrieval-augmented generation and its benefits in about 150 words."


async def one(client):
    start = time.perf_counter()
    ttft, tokens = None, 0
    stream = await client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": PROMPT}], max_tokens=256,
        stream=True, stream_options={"include_usage": True})
    async for ev in stream:
        if ev.choices and ev.choices[0].delta.content and ttft is None:
            ttft = time.perf_counter() - start
        if ev.usage:
            tokens = ev.usage.completion_tokens
    return ttft, time.perf_counter() - start, tokens


async def main(n):
    client = AsyncOpenAI(base_url=BASE, api_key="x")
    t0 = time.perf_counter()
    results = await asyncio.gather(*[one(client) for _ in range(n)])
    wall = time.perf_counter() - t0
    ttfts = sorted(r[0] for r in results)
    lats = sorted(r[1] for r in results)
    toks = sum(r[2] for r in results)
    p95 = lambda xs: xs[int(0.95 * (len(xs) - 1))]
    print(f"Requests: {n}  | wall: {wall:.2f}s")
    print(f"Output throughput: {toks / wall:.1f} tokens/s")
    print(f"TTFT mean/P95: {statistics.mean(ttfts):.3f}s / {p95(ttfts):.3f}s")
    print(f"Latency mean/P95: {statistics.mean(lats):.2f}s / {p95(lats):.2f}s")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--concurrency", type=int, default=32)
    asyncio.run(main(ap.parse_args().concurrency))


### `scripts/run_checks.py` - end-to-end checks

In [ ]:
%%writefile scripts/run_checks.py
"""End-to-end check runner for a GPU box (Kaggle/Colab): starts vLLM, Qdrant and the API,
builds the index, then exercises every agent path, /ask-image, /health and the benchmark.

python scripts/run_checks.py            (from the project root, after pip install)
"""
import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import requests

MODEL = os.getenv("VLLM_MODEL", "Qwen/Qwen2.5-VL-7B-Instruct-AWQ")
API = "http://localhost:8081"
results = []


def check(name, ok, detail=""):
    results.append((name, ok, detail))
    print(f"[{'PASS' if ok else 'FAIL'}] {name} {detail}", flush=True)


def start(cmd, log):
    # own session + no inherited stdout, so the notebook cell can finish while servers keep running
    return subprocess.Popen(f"exec {cmd} > {log} 2>&1", shell=True, start_new_session=True,
                            stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def wait_for(url, minutes):
    for _ in range(minutes * 6):
        try:
            if requests.get(url, timeout=3).ok:
                return True
        except Exception:
            pass
        time.sleep(10)
    return False


def ask(q):
    t = time.time()
    r = requests.post(f"{API}/ask", json={"question": q}, timeout=900).json()
    print(f"\nQ: {q} ({time.time() - t:.1f}s)\n{json.dumps(r, indent=2)}", flush=True)
    return r


# 1. vLLM
start(f"vllm serve {MODEL} --port 8000 --max-model-len 4096 --gpu-memory-utilization 0.80 "
      "--quantization awq --dtype float16 --limit-mm-per-prompt '{\"image\":1}' --enforce-eager", "vllm.log")
check("vLLM server up", wait_for("http://localhost:8000/v1/models", 20))

# 2. Qdrant
if not Path("qdrant").exists():
    url = "https://github.com/qdrant/qdrant/releases/download/v1.12.4/qdrant-x86_64-unknown-linux-gnu.tar.gz"
    urllib.request.urlretrieve(url, "qdrant.tgz")
    subprocess.run("tar -xzf qdrant.tgz", shell=True, check=True)
start("./qdrant", "qdrant.log")
check("Qdrant up", wait_for("http://localhost:6333", 2))

# 3. Ingestion
subprocess.run([sys.executable, "scripts/download_docs.py"], check=True)
ing = subprocess.run([sys.executable, "-m", "app.ingest"], capture_output=True, text=True)
summary = [l for l in ing.stdout.splitlines() if "chunks" in l or "failed" in l]
print("\n".join(summary))
check("Ingestion", ing.returncode == 0 and Path("data/bm25.pkl").exists(), summary[0] if summary else ing.stderr[-500:])

# 4. API
start("uvicorn app.main:app --port 8081", "api.log")
ok = wait_for(f"{API}/health", 3)
health = requests.get(f"{API}/health").json() if ok else {}
check("/health", health.get("status") == "ok", str(health))

# 5. Agent paths
r = ask("What problem does PagedAttention solve in LLM serving?")
check("Document path (grade -> answer -> verify)", "verify:pass" in r["agent_path"] and "[Source:" in r["answer"], str(r["agent_path"]))
r = ask("What does Figure 1 of the Transformer paper show?")
check("Figure question answered from a figure chunk", any(s["type"] == "figure" for s in r["sources"]), str(r["sources"]))
r = ask("What are the two pre-training tasks of BERT?")
check("BERT question (known weak case)", "verify:pass" in r["agent_path"], r["answer"][:120])
r = ask("Hi there!")
check("Chit-chat route", r["agent_path"][0] == "router:chitchat", str(r["agent_path"]))
r = ask("Who won the 2022 FIFA World Cup?")
check("Out-of-scope -> fallback", r["agent_path"][-1] == "fallback" and "couldn't find" in r["answer"], str(r["agent_path"]))
r = ask("What learning rate did the LoRA paper use to fine-tune GPT-5?")
check("Plausible-but-unanswerable -> verify fail -> fallback", "couldn't find" in r["answer"], str(r["agent_path"]))
r = ask("What quantum error-correction code does the ReAct paper propose?")
check("Nothing relevant -> rewrite x2 -> fallback", r["agent_path"].count("rewrite") == 2 and r["agent_path"][-1] == "fallback", str(r["agent_path"]))
r = ask("")
# empty question should be rejected by the API (400) -- ask() returns the error body
check("Empty question rejected", "detail" in r, str(r))

# 6. Visual Q&A
import pymupdf
pymupdf.open("data/pdfs/attention_is_all_you_need.pdf")[2].get_pixmap(dpi=100).save("page.png")
with open("page.png", "rb") as f:
    r = requests.post(f"{API}/ask-image", data={"question": "Which components are in this diagram?"},
                      files={"image": ("page.png", f, "image/png")}, timeout=600).json()
print("\n/ask-image:", r.get("answer", r)[:400])
check("/ask-image", "answer" in r and len(r["answer"]) > 50)

# 7. Benchmark
b = subprocess.run([sys.executable, "scripts/benchmark.py", "--concurrency", "32"], capture_output=True, text=True)
print("\n" + b.stdout)
check("Benchmark", b.returncode == 0, b.stdout.strip().replace("\n", " | ") or b.stderr[-300:])

print("\n===== SUMMARY =====")
for name, ok, _ in results:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"{sum(ok for _, ok, _ in results)}/{len(results)} passed")


### `eval/eval.py` - evaluation

In [ ]:
%%writefile eval/eval.py
"""Bonus: LLM-as-judge eval. Answers come from our vLLM-backed /ask API;
Gemini is used ONLY as the judge (allowed by the assignment rules).

The two steps can run on different machines (answers on the GPU box, judging anywhere):
  python eval/eval.py --collect eval/answers.json   (API must be running on :8081)
  python eval/eval.py --judge   eval/answers.json   (needs GEMINI_API_KEY)
  python eval/eval.py                                (both in one go)
"""
import argparse
import json
import os

import requests
from dotenv import load_dotenv

load_dotenv()
API = os.getenv("API_URL", "http://localhost:8081")

QUESTIONS = [
    "How many attention heads does the base Transformer use?",
    "What optimizer and learning-rate schedule was used to train the Transformer?",
    "What are the two pre-training tasks of BERT?",
    "What is the difference between RAG-Sequence and RAG-Token?",
    "What encoder does Dense Passage Retrieval use for questions and passages?",
    "Why does LoRA introduce no additional inference latency?",
    "What problem does PagedAttention solve in LLM serving?",
    "What are reflection tokens in Self-RAG?",
    "What does the retrieval evaluator do in Corrective RAG?",
    "What three retrieval functionalities does BGE-M3 support?",
    "How does ReAct combine reasoning and acting?",
    "What is chain-of-thought prompting?",
    "Who won the 2022 FIFA World Cup?",  # should hit fallback
]

JUDGE = """You are grading a RAG system. Question: {q}
Answer: {a}
Retrieved sources: {s}

Return JSON only: {{"faithfulness": 1-5, "relevance": 1-5, "reason": "..."}}
faithfulness = answer only makes claims plausibly supported by the cited papers (an honest "not found" is 5).
relevance = answer addresses the question (an honest "not found" for an out-of-scope question is 5)."""


def collect(path):
    rows = []
    for q in QUESTIONS:
        r = requests.post(f"{API}/ask", json={"question": q}, timeout=900).json()
        rows.append({"question": q, **r})
        print(f"{r['agent_path'][-1]:>12} | {q}")
    json.dump(rows, open(path, "w"), indent=2)
    return rows


def judge(rows):
    from google import genai
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    model = os.getenv("JUDGE_MODEL", "gemini-3.8-flash")
    for row in rows:
        g = client.models.generate_content(
            model=model, contents=JUDGE.format(q=row["question"], a=row["answer"], s=row["sources"]))
        text = g.text.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
        row.update(json.loads(text))
        print(f"[F{row['faithfulness']} R{row['relevance']}] {row['question']}")
    n = len(rows)
    print(f"\nMean faithfulness: {sum(r['faithfulness'] for r in rows) / n:.2f}/5")
    print(f"Mean relevance:    {sum(r['relevance'] for r in rows) / n:.2f}/5")
    json.dump(rows, open("eval/results.json", "w"), indent=2)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--collect", metavar="OUT")
    ap.add_argument("--judge", metavar="IN")
    args = ap.parse_args()
    if args.collect:
        collect(args.collect)
    elif args.judge:
        judge(json.load(open(args.judge)))
    else:
        judge(collect("eval/answers.json"))


## Install
I uninstall torchaudio because the version in the Kaggle image was built for a different CUDA version than the torch that vLLM installs, and it breaks the import.

In [ ]:
!pip install -q vllm 2>&1 | tail -n 1
!pip install -q -r requirements.txt 2>&1 | tail -n 1
!pip uninstall -y -q torchaudio
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())"

## Start everything and run the checks
`run_checks.py` starts vLLM, Qdrant and the API, builds the index (including figure descriptions), and then tests each agent path, `/ask-image` and the benchmark. vLLM takes 5-10 min to come up on a T4.

In [ ]:
!python -u scripts/run_checks.py 2>&1 | tee /kaggle/working/checks.log | grep -E 'PASS|FAIL|passed|Requests|throughput|TTFT|Latency'

## Demo
`PATH` is the route the request took through the graph.

In [ ]:
import requests, textwrap

def ask(question):
    r = requests.post("http://localhost:8081/ask", json={"question": question}, timeout=900).json()
    print(f"Q: {question}")
    print("PATH:", " -> ".join(r["agent_path"]))
    print("ANSWER:")
    print(textwrap.fill(r["answer"], 100))
    if r["sources"]:
        print("SOURCES:", sorted({f"{s['source']} p.{s['page']} ({s['type']})" for s in r["sources"]}))
    print("-" * 100)

ask("What problem does PagedAttention solve in LLM serving?")                  # normal question
ask("What does Figure 1 of the Transformer paper show?")                        # figure
ask("Hi there!")                                                               # small talk
ask("Who won the 2022 FIFA World Cup?")                                        # off-topic
ask("What quantum error-correction code does the ReAct paper propose?")         # nothing relevant, should rewrite twice then give up

## /ask-image

In [ ]:
import pymupdf
pymupdf.open("data/pdfs/attention_is_all_you_need.pdf")[2].get_pixmap(dpi=100).save("page.png")
with open("page.png", "rb") as f:
    r = requests.post("http://localhost:8081/ask-image",
                      data={"question": "Which components are shown in this diagram?"},
                      files={"image": ("page.png", f, "image/png")}, timeout=600).json()
print(textwrap.fill(r["answer"], 100))

## Evaluation answers
This collects answers to the 13 eval questions into `answers.json`. I scored them with Gemini on my own machine (`python eval/eval.py --judge answers.json`), so no API key is needed here. The result was faithfulness 4.62/5 and relevance 4.69/5. The README goes through the weak answers.

In [ ]:
!python eval/eval.py --collect /kaggle/working/answers.json